In [3]:
import os
import joblib
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [ ]:
X_train = pd.read_csv('data/X_train_pre.csv')
X_val   = pd.read_csv('data/X_val_pre.csv')
X_test  = pd.read_csv('data/X_test_pre.csv')
y_train = pd.read_csv('data/y_train.csv').squeeze('columns')
y_val   = pd.read_csv('data/y_val.csv').squeeze('columns')
y_test  = pd.read_csv('data/y_test.csv').squeeze('columns')

In [ ]:
# KNN: tune k and weights on the validation set
knn_val = {}
for weights in ["uniform", "distance"]:
    for k in [1, 3, 5, 7, 9, 11, 15, 21]:
        model = KNeighborsRegressor(n_neighbors=k, weights=weights).fit(X_train, y_train)
        pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, pred)
        mae = mean_absolute_error(y_val, pred)
        r2 = r2_score(y_val, pred)
        knn_val[(k, weights)] = rmse
        print(f"k={k:<3} weights={weights:<9} RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.4f}")


best_k, best_w = min(knn_val, key=lambda kw: (knn_val[kw], -kw[0], kw[1] != "uniform"))
print("Best KNN:", best_k, best_w)

k=1   weights=uniform   RMSE=6521.37  MAE=3357.10  R2=0.7514
k=3   weights=uniform   RMSE=5940.32  MAE=3488.53  R2=0.7937
k=5   weights=uniform   RMSE=6150.66  MAE=3799.72  R2=0.7789
k=7   weights=uniform   RMSE=6539.67  MAE=4016.15  R2=0.7500
k=9   weights=uniform   RMSE=6763.80  MAE=4128.52  R2=0.7326
k=11  weights=uniform   RMSE=6944.66  MAE=4245.84  R2=0.7181
k=15  weights=uniform   RMSE=7045.47  MAE=4287.06  R2=0.7099
k=21  weights=uniform   RMSE=7274.13  MAE=4338.90  R2=0.6907
k=1   weights=distance  RMSE=6521.37  MAE=3357.10  R2=0.7514
k=3   weights=distance  RMSE=6047.83  MAE=3466.52  R2=0.7862
k=5   weights=distance  RMSE=6141.02  MAE=3669.68  R2=0.7796
k=7   weights=distance  RMSE=6388.58  MAE=3847.49  R2=0.7614
k=9   weights=distance  RMSE=6533.67  MAE=3903.67  R2=0.7505
k=11  weights=distance  RMSE=6678.38  MAE=4024.30  R2=0.7393
k=15  weights=distance  RMSE=6789.84  MAE=4085.50  R2=0.7305
k=21  weights=distance  RMSE=6951.50  MAE=4132.39  R2=0.7175
Best KNN: 3 uniform


In [ ]:
knn_model = KNeighborsRegressor(n_neighbors=best_k, weights=best_w)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print("KNN Regressor - Test")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_knn):.2f}")
print(f"MAE : {mean_absolute_error(y_test, y_pred_knn):.2f}")
print(f"R2  : {r2_score(y_test, y_pred_knn):.4f}")

KNN Regressor - Test
RMSE: 5901.32
MAE : 3481.53
R2  : 0.7959


In [10]:
os.makedirs('../models', exist_ok=True)
joblib.dump(knn_model, '../models/knn_regressor.pkl')

['../models/knn_regressor.pkl']